# 05 — Large Scale Join Benchmark

Benchmark-oriented comparison of join strategies on larger synthetic datasets.

## 00 — Setup

```text
SparkSession
    │
    ├── warehouse: /tmp/spark-warehouse
    ├── shuffle partitions: 64
    ├── AQE toggled per benchmark
    └── benchmark action: count()
```

In [1]:
from pyspark.sql import SparkSession, functions as F
import time

spark = (
    SparkSession.builder
    .appName("large_scale_join_benchmark")
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse")
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "64")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

spark


## 01 — Benchmark helper

```text
DataFrame
   │
   ├── action: count()
   ├── rows returned
   └── elapsed seconds
```

In [2]:
benchmark_results = []

def benchmark(label, df):
    start = time.time()
    rows = df.count()
    seconds = round(time.time() - start, 3)
    benchmark_results.append((label, rows, seconds))
    print(label, rows, seconds)
    return rows, seconds

def show_results():
    return spark.createDataFrame(benchmark_results, ["case", "rows", "seconds"]).orderBy("seconds")


## 02 — Large fact table and dimensions

```text
fact_sales_large
  sale_id
  customer_id ─────────────┐
  product_id               │
  store_id                 │
  amount                   │
                           ├── join key
customers_large            │
  customer_id ─────────────┘
  country
  segment

small_segments
  segment ──────────────── support model
```

In [3]:
FACT_ROWS = 5_000_000
CUSTOMER_ROWS = 500_000

fact_sales_large = (
    spark.range(0, FACT_ROWS)
    .withColumnRenamed("id", "sale_id")
    .withColumn("customer_id", (F.col("sale_id") % CUSTOMER_ROWS).cast("long"))
    .withColumn("product_id", (F.col("sale_id") % 100_000).cast("long"))
    .withColumn("store_id", (F.col("sale_id") % 2_000).cast("long"))
    .withColumn("amount", ((F.col("sale_id") % 500) + 10).cast("double"))
)

customers_large = (
    spark.range(0, CUSTOMER_ROWS)
    .withColumnRenamed("id", "customer_id")
    .withColumn("country", F.expr("CASE WHEN customer_id % 5 = 0 THEN 'SK' WHEN customer_id % 5 = 1 THEN 'CZ' WHEN customer_id % 5 = 2 THEN 'DE' WHEN customer_id % 5 = 3 THEN 'AT' ELSE 'PL' END"))
    .withColumn("segment", F.expr("CASE WHEN customer_id % 4 = 0 THEN 'enterprise' WHEN customer_id % 4 = 1 THEN 'smb' WHEN customer_id % 4 = 2 THEN 'consumer' ELSE 'public' END"))
)

small_segments = spark.createDataFrame(
    [
        ("enterprise", "high_touch"),
        ("smb", "standard"),
        ("consumer", "self_service"),
        ("public", "managed"),
    ],
    ["segment", "support_model"]
)

fact_sales_large.count(), customers_large.count(), small_segments.count()


(5000000, 500000, 4)

## 03 — Baseline shuffle join

```text
fact_sales_large ── shuffle(customer_id) ──┐
                                           ├── join
customers_large  ── shuffle(customer_id) ──┘

Both sides are large enough to force network shuffle.
```

In [4]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

baseline_shuffle = fact_sales_large.join(customers_large, "customer_id")
baseline_shuffle.explain(mode="simple")
benchmark("01_baseline_shuffle_no_aqe", baseline_shuffle)


== Physical Plan ==
*(5) Project [customer_id#2L, sale_id#1L, product_id#3L, store_id#4L, amount#5, country#8, segment#9]
+- *(5) SortMergeJoin [customer_id#2L], [customer_id#7L], Inner
   :- *(2) Sort [customer_id#2L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#2L, 64), ENSURE_REQUIREMENTS, [plan_id=135]
   :     +- *(1) Project [id#0L AS sale_id#1L, (id#0L % 500000) AS customer_id#2L, (id#0L % 100000) AS product_id#3L, (id#0L % 2000) AS store_id#4L, cast(((id#0L % 500) + 10) as double) AS amount#5]
   :        +- *(1) Filter isnotnull((id#0L % 500000))
   :           +- *(1) Range (0, 5000000, step=1, splits=2)
   +- *(4) Sort [customer_id#7L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(customer_id#7L, 64), ENSURE_REQUIREMENTS, [plan_id=141]
         +- *(3) Project [id#6L AS customer_id#7L, CASE WHEN ((id#6L % 5) = 0) THEN SK WHEN ((id#6L % 5) = 1) THEN CZ WHEN ((id#6L % 5) = 2) THEN DE WHEN ((id#6L % 5) = 3) THEN AT ELSE PL END AS count

[Stage 11:======================================================> (62 + 2) / 64]

01_baseline_shuffle_no_aqe 5000000 3.4


(5000000, 3.4)

## 04 — Broadcast join with large dimension

```text
customers_large
      │
      ├── broadcast to every executor
      ▼
local hash lookup against fact_sales_large partitions

This can be faster, but it increases executor memory pressure.
```

In [5]:
broadcast_large_dim = fact_sales_large.join(F.broadcast(customers_large), "customer_id")
broadcast_large_dim.explain(mode="simple")
benchmark("02_forced_broadcast_large_dimension", broadcast_large_dim)


== Physical Plan ==
*(2) Project [customer_id#2L, sale_id#1L, product_id#3L, store_id#4L, amount#5, country#8, segment#9]
+- *(2) BroadcastHashJoin [customer_id#2L], [customer_id#7L], Inner, BuildRight, false
   :- *(2) Project [id#0L AS sale_id#1L, (id#0L % 500000) AS customer_id#2L, (id#0L % 100000) AS product_id#3L, (id#0L % 2000) AS store_id#4L, cast(((id#0L % 500) + 10) as double) AS amount#5]
   :  +- *(2) Filter isnotnull((id#0L % 500000))
   :     +- *(2) Range (0, 5000000, step=1, splits=2)
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=290]
      +- *(1) Project [id#6L AS customer_id#7L, CASE WHEN ((id#6L % 5) = 0) THEN SK WHEN ((id#6L % 5) = 1) THEN CZ WHEN ((id#6L % 5) = 2) THEN DE WHEN ((id#6L % 5) = 3) THEN AT ELSE PL END AS country#8, CASE WHEN ((id#6L % 4) = 0) THEN enterprise WHEN ((id#6L % 4) = 1) THEN smb WHEN ((id#6L % 4) = 2) THEN consumer ELSE public END AS segment#9]
         +- *(1) Range (0, 500000, step=1, sp

(5000000, 0.645)

## 05 — Tiny dimension broadcast

```text
customers_large ── join(segment) ── small_segments
                                  │
                                  └── tiny table broadcast

Ideal broadcast case.
```

In [6]:
tiny_dimension_join = customers_large.join(F.broadcast(small_segments), "segment")
tiny_dimension_join.explain(mode="simple")
benchmark("03_tiny_dimension_broadcast", tiny_dimension_join)


== Physical Plan ==
*(2) Project [segment#9, customer_id#7L, country#8, support_model#11]
+- *(2) BroadcastHashJoin [segment#9], [segment#10], Inner, BuildRight, false
   :- *(2) Project [id#6L AS customer_id#7L, CASE WHEN ((id#6L % 5) = 0) THEN SK WHEN ((id#6L % 5) = 1) THEN CZ WHEN ((id#6L % 5) = 2) THEN DE WHEN ((id#6L % 5) = 3) THEN AT ELSE PL END AS country#8, CASE WHEN ((id#6L % 4) = 0) THEN enterprise WHEN ((id#6L % 4) = 1) THEN smb WHEN ((id#6L % 4) = 2) THEN consumer ELSE public END AS segment#9]
   :  +- *(2) Range (0, 500000, step=1, splits=2)
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=397]
      +- *(1) Filter isnotnull(segment#10)
         +- *(1) Scan ExistingRDD[segment#10,support_model#11]


03_tiny_dimension_broadcast 500000 0.435


(500000, 0.435)

## 06 — AQE enabled shuffle join

```text
initial shuffle plan
        │
        ├── runtime statistics
        ├── partition coalescing
        └── adaptive final plan
```

In [7]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

aqe_shuffle = fact_sales_large.join(customers_large, "customer_id")
aqe_shuffle.explain(mode="formatted")
benchmark("04_shuffle_with_aqe", aqe_shuffle)
aqe_shuffle.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- SortMergeJoin Inner (10)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Project (3)
      :        +- Filter (2)
      :           +- Range (1)
      +- Sort (9)
         +- Exchange (8)
            +- Project (7)
               +- Range (6)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 5000000, step=1, splits=Some(2))

(2) Filter
Input [1]: [id#0L]
Condition : isnotnull((id#0L % 500000))

(3) Project
Output [5]: [id#0L AS sale_id#1L, (id#0L % 500000) AS customer_id#2L, (id#0L % 100000) AS product_id#3L, (id#0L % 2000) AS store_id#4L, cast(((id#0L % 500) + 10) as double) AS amount#5]
Input [1]: [id#0L]

(4) Exchange
Input [5]: [sale_id#1L, customer_id#2L, product_id#3L, store_id#4L, amount#5]
Arguments: hashpartitioning(customer_id#2L, 64), ENSURE_REQUIREMENTS, [plan_id=508]

(5) Sort
Input [5]: [sale_id#1L, customer_id#2L, product_id#3L, store_id#4L, amount#5]
Arguments: [customer_id#2L ASC NULLS FIRST]

[Stage 23:=============================>                            (1 + 1) / 2]

04_shuffle_with_aqe 5000000 1.965
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- SortMergeJoin Inner (10)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Project (3)
      :        +- Filter (2)
      :           +- Range (1)
      +- Sort (9)
         +- Exchange (8)
            +- Project (7)
               +- Range (6)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 5000000, step=1, splits=Some(2))

(2) Filter
Input [1]: [id#0L]
Condition : isnotnull((id#0L % 500000))

(3) Project
Output [5]: [id#0L AS sale_id#1L, (id#0L % 500000) AS customer_id#2L, (id#0L % 100000) AS product_id#3L, (id#0L % 2000) AS store_id#4L, cast(((id#0L % 500) + 10) as double) AS amount#5]
Input [1]: [id#0L]

(4) Exchange
Input [5]: [sale_id#1L, customer_id#2L, product_id#3L, store_id#4L, amount#5]
Arguments: hashpartitioning(customer_id#2L, 64), ENSURE_REQUIREMENTS, [plan_id=508]

(5) Sort
Input [5]: [sale_id#1L, customer_id#2L, product_id#3L, store_id#4L, amount#5]
Arguments

## 07 — Filtered dimension with AQE broadcast opportunity

```text
customers_large
      │
      ├── filter country = 'SK'
      ▼
smaller runtime relation
      │
      └── AQE may choose better execution after statistics
```

In [8]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", 64 * 1024 * 1024)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

filtered_customers = customers_large.filter(F.col("country") == "SK")
filtered_aqe_join = fact_sales_large.join(filtered_customers, "customer_id")
filtered_aqe_join.explain(mode="formatted")
benchmark("05_filtered_dimension_aqe", filtered_aqe_join)
filtered_aqe_join.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (13)
+- Project (12)
   +- SortMergeJoin Inner (11)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Project (3)
      :        +- Filter (2)
      :           +- Range (1)
      +- Sort (10)
         +- Exchange (9)
            +- Project (8)
               +- Filter (7)
                  +- Range (6)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 5000000, step=1, splits=Some(2))

(2) Filter
Input [1]: [id#0L]
Condition : (CASE WHEN (((id#0L % 500000) % 5) = 0) THEN true WHEN (((id#0L % 500000) % 5) = 1) THEN false WHEN (((id#0L % 500000) % 5) = 2) THEN false WHEN (((id#0L % 500000) % 5) = 3) THEN false ELSE false END AND isnotnull((id#0L % 500000)))

(3) Project
Output [5]: [id#0L AS sale_id#1L, (id#0L % 500000) AS customer_id#2L, (id#0L % 100000) AS product_id#3L, (id#0L % 2000) AS store_id#4L, cast(((id#0L % 500) + 10) as double) AS amount#5]
Input [1]: [id#0L]

(4) Exchange
Input [5]: [sale_id#1L, customer_id#2L, product_id#3L, 

## 08 — Skewed large join

```text
normal events
  customer_id distributed
        │
        ├── union
        ▼
hot events
  customer_id = 1 repeated many times

One key dominates the shuffle.
```

In [9]:
normal_skew_fact = (
    spark.range(0, 1_500_000)
    .withColumnRenamed("id", "sale_id")
    .withColumn("customer_id", ((F.col("sale_id") % (CUSTOMER_ROWS - 1)) + 1).cast("long"))
    .withColumn("amount", ((F.col("sale_id") % 500) + 10).cast("double"))
)

hot_skew_fact = (
    spark.range(0, 3_500_000)
    .withColumnRenamed("id", "sale_id")
    .withColumn("customer_id", F.lit(1).cast("long"))
    .withColumn("amount", ((F.col("sale_id") % 500) + 10).cast("double"))
)

skewed_fact = normal_skew_fact.unionByName(hot_skew_fact)

skewed_fact.groupBy("customer_id").count().orderBy(F.desc("count")).show(5)


+-----------+-------+
|customer_id|  count|
+-----------+-------+
|          1|3500004|
|          3|      4|
|          2|      4|
|         41|      3|
|         93|      3|
+-----------+-------+
only showing top 5 rows


## 09 — Partition metrics for skew

```text
repartition(customer_id)
        │
        ▼
partition_id → row count
        │
        └── largest partition reveals hot key pressure
```

In [10]:
skew_partition_metrics = (
    skewed_fact
    .repartition(64, "customer_id")
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy(F.desc("count"))
)

skew_partition_metrics.show(20, truncate=False)


+------------+-------+
|partition_id|count  |
+------------+-------+
|29          |3523362|
|61          |24072  |
|1           |24054  |
|13          |24054  |
|31          |23925  |
|11          |23898  |
|23          |23886  |
|30          |23841  |
|59          |23808  |
|39          |23760  |
|10          |23748  |
|43          |23697  |
|62          |23697  |
|0           |23688  |
|56          |23679  |
|35          |23674  |
|4           |23613  |
|7           |23613  |
|16          |23604  |
|36          |23568  |
+------------+-------+
only showing top 20 rows


## 10 — Skewed join with AQE

```text
skewed shuffle partition
        │
        ├── AQE detects abnormal size
        ├── split skewed partition
        └── reduce straggler impact
```

In [11]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "2")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "1024")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

skewed_aqe_join = skewed_fact.join(customers_large, "customer_id")
skewed_aqe_join.explain(mode="formatted")
benchmark("06_skewed_join_with_aqe", skewed_aqe_join)
skewed_aqe_join.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (15)
+- Project (14)
   +- SortMergeJoin Inner (13)
      :- Sort (8)
      :  +- Exchange (7)
      :     +- Union (6)
      :        :- Project (3)
      :        :  +- Filter (2)
      :        :     +- Range (1)
      :        +- Project (5)
      :           +- Range (4)
      +- Sort (12)
         +- Exchange (11)
            +- Project (10)
               +- Range (9)


(1) Range
Output [1]: [id#86L]
Arguments: Range (0, 1500000, step=1, splits=Some(2))

(2) Filter
Input [1]: [id#86L]
Condition : isnotnull(((id#86L % 499999) + 1))

(3) Project
Output [3]: [id#86L AS sale_id#87L, ((id#86L % 499999) + 1) AS customer_id#88L, cast(((id#86L % 500) + 10) as double) AS amount#89]
Input [1]: [id#86L]

(4) Range
Output [1]: [id#90L]
Arguments: Range (0, 3500000, step=1, splits=Some(2))

(5) Project
Output [3]: [id#90L AS sale_id#91L, 1 AS customer_id#92L, cast(((id#90L % 500) + 10) as double) AS amount#93]
Input [1]: [id#90L]

(6) Union

(7) Exchange

06_skewed_join_with_aqe 5000000 1.329
== Physical Plan ==
AdaptiveSparkPlan (15)
+- Project (14)
   +- SortMergeJoin Inner (13)
      :- Sort (8)
      :  +- Exchange (7)
      :     +- Union (6)
      :        :- Project (3)
      :        :  +- Filter (2)
      :        :     +- Range (1)
      :        +- Project (5)
      :           +- Range (4)
      +- Sort (12)
         +- Exchange (11)
            +- Project (10)
               +- Range (9)


(1) Range
Output [1]: [id#86L]
Arguments: Range (0, 1500000, step=1, splits=Some(2))

(2) Filter
Input [1]: [id#86L]
Condition : isnotnull(((id#86L % 499999) + 1))

(3) Project
Output [3]: [id#86L AS sale_id#87L, ((id#86L % 499999) + 1) AS customer_id#88L, cast(((id#86L % 500) + 10) as double) AS amount#89]
Input [1]: [id#86L]

(4) Range
Output [1]: [id#90L]
Arguments: Range (0, 3500000, step=1, splits=Some(2))

(5) Project
Output [3]: [id#90L AS sale_id#91L, 1 AS customer_id#92L, cast(((id#90L % 500) + 10) as double) AS amount#93]
Input 

## 11 — Manual salting benchmark

```text
skewed_fact hot customer_id = 1
        │
        ├── random salt 0..N-1
        ▼
customers_large hot customer_id = 1
        │
        ├── duplicate across salt 0..N-1
        ▼
join on (customer_id, salt)
```

In [12]:
SALT_BUCKETS = 8

salted_fact = skewed_fact.withColumn(
    "salt",
    F.when(F.col("customer_id") == 1, (F.rand(seed=42) * SALT_BUCKETS).cast("int")).otherwise(F.lit(0))
)

salt_array = F.when(
    F.col("customer_id") == 1,
    F.array(*[F.lit(i) for i in range(SALT_BUCKETS)])
).otherwise(F.array(F.lit(0)))

salted_customers = customers_large.withColumn("salt", F.explode(salt_array))

spark.conf.set("spark.sql.adaptive.enabled", "false")

salted_join = salted_fact.join(salted_customers, ["customer_id", "salt"])
salted_join.explain(mode="simple")
benchmark("07_manual_salting", salted_join)


== Physical Plan ==
*(7) Project [customer_id#88L, salt#133, sale_id#87L, amount#89, country#8, segment#9]
+- *(7) SortMergeJoin [customer_id#88L, salt#133], [customer_id#7L, salt#135], Inner
   :- *(4) Sort [customer_id#88L ASC NULLS FIRST, salt#133 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#88L, salt#133, 64), ENSURE_REQUIREMENTS, [plan_id=1463]
   :     +- *(3) Filter (isnotnull(customer_id#88L) AND isnotnull(salt#133))
   :        +- *(3) Project [sale_id#87L, customer_id#88L, amount#89, CASE WHEN (customer_id#88L = 1) THEN cast((rand(42) * 8.0) as int) ELSE 0 END AS salt#133]
   :           +- Union
   :              :- *(1) Project [id#86L AS sale_id#87L, ((id#86L % 499999) + 1) AS customer_id#88L, cast(((id#86L % 500) + 10) as double) AS amount#89]
   :              :  +- *(1) Range (0, 1500000, step=1, splits=2)
   :              +- *(2) Project [id#90L AS sale_id#91L, 1 AS customer_id#92L, cast(((id#90L % 500) + 10) as double) AS amount#93]
   : 

[Stage 57:==========================================>             (49 + 2) / 64]

07_manual_salting 5000000 2.089


(5000000, 2.089)

## 12 — Benchmark results

```text
all cases
   │
   ├── rows
   ├── seconds
   └── sorted by runtime
```

In [13]:
results_df = show_results()
results_df.show(truncate=False)


+-----------------------------------+-------+-------+
|case                               |rows   |seconds|
+-----------------------------------+-------+-------+
|03_tiny_dimension_broadcast        |500000 |0.435  |
|02_forced_broadcast_large_dimension|5000000|0.645  |
|05_filtered_dimension_aqe          |1000000|0.655  |
|06_skewed_join_with_aqe            |5000000|1.329  |
|04_shuffle_with_aqe                |5000000|1.965  |
|07_manual_salting                  |5000000|2.089  |
|01_baseline_shuffle_no_aqe         |5000000|3.4    |
+-----------------------------------+-------+-------+



## 13 — Decision table

```text
large-scale join
    │
    ├── small dimension       → broadcast
    ├── both sides large      → shuffle / sort-merge
    ├── filtered dimension    → AQE may help
    ├── skewed key            → AQE skew or salting
    └── repeated benchmarks   → compare plans + runtime, not runtime only
```

In [14]:
decision_rows = [
    ("small dimension", "broadcast", "Best when dimension clearly fits executor memory"),
    ("large dimension", "shuffle join", "Safer default when broadcast would stress memory"),
    ("filtered dimension", "AQE", "Runtime statistics can improve the plan"),
    ("single hot key", "manual salting", "Deterministic split of one dominant key"),
    ("unknown skew", "AQE skew handling", "Automatic split based on shuffle partition size"),
]

spark.createDataFrame(decision_rows, ["case", "strategy", "reason"]).show(truncate=False)


+------------------+-----------------+------------------------------------------------+
|case              |strategy         |reason                                          |
+------------------+-----------------+------------------------------------------------+
|small dimension   |broadcast        |Best when dimension clearly fits executor memory|
|large dimension   |shuffle join     |Safer default when broadcast would stress memory|
|filtered dimension|AQE              |Runtime statistics can improve the plan         |
|single hot key    |manual salting   |Deterministic split of one dominant key         |
|unknown skew      |AQE skew handling|Automatic split based on shuffle partition size |
+------------------+-----------------+------------------------------------------------+

